<h3> Cleanning, classfying, and counting sentiment score distribution for the original headline

In [ ]:
import pandas as pd
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize.toktok import ToktokTokenizer
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer 
import torch 
import os

# Downloads stopwords
nltk.download('stopwords')

# Loads data
path = ''
years = range(2015, 2020)
featuresdfs = [pd.read_csv(f'{path}{year}.csv') for year in years]
featuresdf = pd.concat(featuresdfs, ignore_index=True)
headlinedf = featuresdf.iloc[:, :2]

# Initializes stopword list and tokenizer
stopWordlst = set(stopwords.words('english'))
textTokenizer = ToktokTokenizer()

# Defines a cleaning function
def preprocess_text(text):
    text = text.lower() # Lowercases
    text = re.sub(r'\[.*?\]', '', text) # Removes square brackets and content
    text = re.sub(r"[,’\.!?:\-;()–']", '', text) # Removes punctuation
    text = re.sub(r'\s+', ' ', text).strip() # Removes extra spaces
    tokens = textTokenizer.tokenize(text) # Tokenizes
    filtered_tokens = [token for token in tokens if token not in stopWordlst] # Removes stopwords
    return ' '.join(filtered_tokens)

# Applies the function to the 'Original Headline' column
headlinedf['Cleaned Headline'] = headlinedf['Original Headline'].astype(str).apply(preprocess_text)

# Saves the cleaned DataFrame to a CSV file
# headlinedf.to_csv('Cleaned Original Headline.csv', index=False)

# Retrieves the two configured files for the CrudeBert
configPath = './crude_bert_config.json' 
modelPath = './crude_bert_model.bin'
# Loads the configuration
config = AutoConfig.from_pretrained(configPath)
# Creates the model from the configuration
model = AutoModelForSequenceClassification.from_config(config)
# Loads the model's state dictionary
state_dict = torch.load(modelPath)
# Inspects keys, if "bert.embeddings.position_ids" is unexpected, remove or adjust it
state_dict.pop("bert.embeddings.position_ids", None)
# Loads the adjusted state dictionary into the model
model.load_state_dict(state_dict, strict=False) # Using strict=False to ignore non-critical mismatches
# Loads the tokenizer
BERTTokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Defines the classification function
def predictionSentiment(texts, model, tokenizer): 
    model.eval() 
    data = [] 
    totalTexts = len(texts)

    for i, text in enumerate(texts):
        print(f"Processing {i+1}/{totalTexts}...", end="\r")
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=64) 
        
        with torch.no_grad(): 
            outputs = model(**inputs) 
            logits = outputs.logits 
            softmax_scores = torch.nn.functional.softmax(logits, dim=-1) 

            pred_label_id = torch.argmax(softmax_scores, dim=-1).item() 
            class_names = ['positive', 'negative', 'neutral'] 
            predicted_label = class_names[pred_label_id]

            # Extract and round probabilities
            softmax_probs = softmax_scores.tolist()[0]
            positive_prob = round(softmax_probs[0], 2)
            negative_prob = round(softmax_probs[1], 2)
            neutral_prob  = round(softmax_probs[2], 2)

            data.append([text, predicted_label, positive_prob, negative_prob, neutral_prob])

    df = pd.DataFrame(data, columns=["Headline", "Classification", "Positive", "Negative", "Neutral"])
    return df

# Calls the classification function
headlineClassificationdf = predictionSentiment(headlinedf['Cleaned Headline'].tolist(), model, BERTTokenizer)

# Adds the 'Date' column from headlinesdf
headlineClassificationdf.insert(0, 'Date', headlinedf['Date'].values)

# Saves the cleaned and classified DataFrame to a CSV file
outputFolder = 'Textual Features (Original Headline)'
os.makedirs(outputFolder, exist_ok=True)
headlineClassificationdf.to_csv(f'{outputFolder}/Cleaned (Original Headline).csv', index=False)


<h3> Removing the outliers using the JSD 

In [ ]:
import pandas as pd
from datetime import timedelta
import numpy as np
from scipy.spatial.distance import jensenshannon
import os

# Loads data
path = 'Textual Features (Original Headline)/Cleaned (Original Headline).csv'
headlineClassificationdf = pd.read_csv(path)
# print(headlineClassificationdf.shape) # Shape [17457, 6]

# Converts 'Date' column to datetime format
headlineClassificationdf['Date'] = pd.to_datetime(headlineClassificationdf['Date'], format='%Y%m%d', errors='coerce')

# Creates rolling 7-day groups from full date range
dateRange = pd.date_range(start="2015-01-01", end="2019-12-31")
groupedRows = []
for groupDate in dateRange:
    startDate = groupDate - timedelta(days=6)
    mask = (headlineClassificationdf['Date'] >= startDate) & (headlineClassificationdf['Date'] <= groupDate)
    group = headlineClassificationdf[mask].copy()
    group['Date Group'] = groupDate
    groupedRows.append(group)

# Combines all groups
headlineClassificationWeeklydf = pd.concat(groupedRows, ignore_index=True)

# Reorder columns to move 'Date Group' next to 'Date'
cols = list(headlineClassificationWeeklydf.columns)
cols.insert(1, cols.pop(cols.index('Date Group')))
headlineClassificationWeeklydf = headlineClassificationWeeklydf[cols]

# Saves the weekly cleaned and classified DataFrame to a CSV file
# headlineClassificationWeeklydf.to_csv('Cleaned Classified Weekly Original Headline.csv', index=False)
print(headlineClassificationWeeklydf.shape) # Shape [122113, 7]

# Defines sentiment columns
sentimentCols = ['Positive', 'Negative', 'Neutral']
# Initializes a list to store JSD values
jsdValues = []
# Groups by 'Date Group'
for groupDate, groupdf in headlineClassificationWeeklydf.groupby('Date Group'):
    groupValues = groupdf[sentimentCols].values
    if len(groupValues) < 2:
        jsdValues.extend([np.nan] * len(groupValues))
        continue
    meanDistribution = np.mean(groupValues, axis=0)
    # Computes JSD between each row and the group mean
    for row_distribution in groupValues:
        jsd = jensenshannon(row_distribution, meanDistribution, base=2)
        jsdValues.append(jsd)

headlineClassificationWeeklydf['JSD'] = jsdValues

# Sets threshold at 99th percentile (global)
threshold = headlineClassificationWeeklydf['JSD'].quantile(0.95)
# Flags outliers
headlineClassificationWeeklydf['Outlier'] = headlineClassificationWeeklydf['JSD'] > threshold
# Counts and remove the outliers
outlierCount = headlineClassificationWeeklydf['Outlier'].sum()
print(f"Number of outliers detected (Top 5% JSD): {outlierCount}")
headlineClassificationWeeklyCleandf = headlineClassificationWeeklydf[~headlineClassificationWeeklydf['Outlier']].reset_index(drop=True)
headlineClassificationWeeklyCleandf = headlineClassificationWeeklyCleandf.drop(columns=['Outlier'])

outputFolder = 'Textual Features (Original Headline)'
os.makedirs(outputFolder, exist_ok=True)
headlineClassificationWeeklyCleandf.to_csv(f'{outputFolder}/Cleaned wo Outline (Original Headline).csv', index=False)

<h3> Choosing one textual feature per day, seven textual features per week (original headline)

In [ ]:
import pandas as pd

filePath = 'Textual Features (Original Headline)/Cleaned wo Outline (Original Headline).csv'
df = pd.read_csv(filePath)

selectedRows = []

for date_group, group_df in df.groupby('Date Group'):
    for date, date_df in group_df.groupby('Date'):
        
        classification_counts = date_df['Classification'].value_counts()
        max_count = classification_counts.max()
        top_classes = classification_counts[classification_counts == max_count].index.tolist()

        for classification in top_classes:
            class_df = date_df[date_df['Classification'] == classification]

            # Choose the correct score column
            if classification.lower() == 'positive':
                top_row = class_df.loc[class_df['Positive'].idxmax()]
            elif classification.lower() == 'negative':
                top_row = class_df.loc[class_df['Negative'].idxmax()]
            elif classification.lower() == 'neutral':
                top_row = class_df.loc[class_df['Neutral'].idxmax()]
            else:
                continue

            selectedRows .append(top_row)

result_df = pd.DataFrame(selectedRows)
outputFolder = 'Textual Features (Original Headline)'
os.makedirs(outputFolder, exist_ok=True)
result_df.to_csv(f'{outputFolder}/Choosen (Original Headline).csv', index=False)

<h3> Word embedding using FinBERT and sentence embedding using average pooling

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import torch
import random
import numpy as np

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Loads the cleaned original headline dataset (have already removed the outliers)
path = 'Textual Features (Original Headline)/Choosen (Original Headline).csv'
cleanedOriginalHeadlinedf = pd.read_csv(path)

# Groups by 'Date Group' and concatenate headlines
cleanedCombinedOriginalHeadlinedf = cleanedOriginalHeadlinedf.groupby('Date Group')['Headline'].apply(lambda x: ' '.join(x)).reset_index()
# Renames columns for clarity (optional)
cleanedCombinedOriginalHeadlinedf.columns = ['Date', 'Headline']
# Saves to CSV
# print(cleanedCombinedOriginalHeadlinedf.shape) # Shape [1826, 2]

# Word Embedding Using FinBERT Process
# Loads FinBERT model (Pre-trained on financial news)
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModel.from_pretrained("ProsusAI/finbert")

# Converts all sequence of headlines/day into a list for FinBERT tokenization
headlineslst = cleanedCombinedOriginalHeadlinedf['Headline'].tolist()

# Tokenizes and encode text using batch_encode_plus
# The function returns a dictionary containing the token IDs and attention masks
encoding = tokenizer.batch_encode_plus( 
    headlineslst, # List of input texts
    padding=True, # Pad to the maximum sequence length
    truncation=True, # Truncate to the maximum sequence length if necessary
    return_tensors='pt', # Return PyTorch tensors
    add_special_tokens=True # Add special tokens CLS and SEP
)

input_ids = encoding['input_ids']  # Token IDs # Shape [1826, 512]
attention_mask = encoding['attention_mask']  # Attention mask # Shape [1826, 512]

# Generates embeddings using FinBERT model with progress tracking
with torch.no_grad():
    outputs = []
    for i in tqdm(range(0, len(input_ids), 32), desc="Processing Headlines", unit="batch"):
        batch_input_ids = input_ids[i:i+32]
        batch_attention_mask = attention_mask[i:i+32]
        
        batch_output = model(batch_input_ids, attention_mask=batch_attention_mask)
        outputs.append(batch_output.last_hidden_state)

# Concatenates results into one tensor
word_embeddings = torch.cat(outputs, dim=0)
print(word_embeddings.shape) # Shape [1826, 512, 768]

# Computes the average of word embeddings to get the sentence embedding
sentence_embedding = word_embeddings.mean(dim=1)  # Average pooling along the sequence length dimension
# Outputs the shape of sentence embeddings
print(sentence_embedding.shape) # Shape [1826, 768]

# Converts tensor to numpy array
sentence_embeddings_np = sentence_embedding.numpy()
# Converts NumPy array to a DataFrame
sentenceEmbeddingsdf = pd.DataFrame(sentence_embeddings_np)
# Adds the date and Original Headline columns
sentenceEmbeddingsdf.insert(0, "Date", cleanedCombinedOriginalHeadlinedf['Date'])
sentenceEmbeddingsdf.insert(1, "Original Headline", headlineslst)
# Saves to a CSV
outputFolder = 'Textual Features (Original Headline)'
os.makedirs(outputFolder, exist_ok=True)
sentenceEmbeddingsdf.to_csv(f'{outputFolder}/Sentence Embedding (Original Headline).csv', index=False)